# DREAM's S2 is a CO2 tax plus a model switch

DREAM's CGE baseline (`main_CGE.gdx`) is solved with `d1switch_integrate_energy_technology = 1`: the calibration
only turns `d1switch_energy_technology` off (`base_model.gmy`, line 20976). Energy use by purpose then follows the
fixed-coefficient branch of `E_qEpj_inNest`:

$$qEpj_{es,e,i,t} = uEpj_{es,e,i,t} \cdot qREes_{es,i,t}$$

DREAM's CO2 tax script sets both switches to 0 (`shock_CO2_tax.gmy`, lines 5397-5398), which selects the
price-elastic branch, with `uEpj` still calibrated for the other one:

$$qEpj_{es,e,i,t} = uEpj_{es,e,i,t} \cdot \left(\frac{pEpj^{marg}_{es,e,i,t}}{pREes_{es,i,t}}\right)^{-eEpj_{es,i}} \cdot qREes_{es,i,t}$$

At the baseline prices the second form does not give the baseline quantities, so the switch alone moves the
economy from 2021, before the tax starts in 2025. ModelFlow reproduces DREAM's S2 (max relative difference
1e-5); a ModelFlow run of the same model with a zero tax isolates the switch:

* **S2 total**: DREAM's S2 against the baseline
* **Switch only**: the zero-tax run against the baseline
* **Tax only**: S2 against the zero-tax run

*Example calculations based on example data. The model is GREU (ModelFlow translation), not DREAM's GreenREFORM-EU.*

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'tools'))
import greu_viz as gv
pd.set_option('display.width', 200)

In [2]:
base = gv.baseline('main_CGE')
s2 = gv.solution('shock_co2tax')          # ModelFlow's S2 = DREAM's S2
zero = gv.solution('own_dose_cge_0')      # same model, CO2 tax 0
YEARS = list(range(2021, 2051))
parts = {'S2 total': (s2, base), 'Switch only': (zero, base), 'Tax only': (s2, zero)}

## The split over time

The switch raises physical energy use (PJ) and CO2e, while the CGE's real energy input `qE` (a volume in
money terms) falls: the price-elastic branch moves energy towards cheap energy goods per PJ.

In [3]:
for df in (base, s2, zero):
    df['ENERGY_PJ'] = gv.energy_pj(df, 'shock_carbon_tax')
vars_ = {'QGDP': 'Real GDP', 'ENERGY_PJ': 'Energy use, PJ', 'QE': 'Real energy inputs (CGE volume)',
         'QEMMTOT__CO2E__UNFCCC': 'CO2e emissions'}
fig = make_subplots(rows=2, cols=2, subplot_titles=list(vars_.values()), horizontal_spacing=0.08,
                    vertical_spacing=0.14)
for j, v in enumerate(vars_, 1):
    for k, (name, (a, b)) in enumerate(parts.items()):
        fig.add_trace(go.Scatter(x=YEARS, y=gv.pct(a.loc[YEARS, v], b.loc[YEARS, v]), mode='lines', name=name,
                                 legendgroup=name, showlegend=(j == 1),
                                 line=dict(color=gv.CAT[k], width=2, dash='dot' if name == 'S2 total' else None),
                                 hovertemplate=name + ' %{x}: %{y:.2f} %<extra></extra>'), row=(j + 1) // 2, col=2 - j % 2)
fig.update_layout(template=gv.TEMPLATE, height=620, hovermode='x unified',
                  title='% difference: S2 total = switch only + tax only (approximately)')
fig.update_yaxes(zeroline=True, zerolinecolor='#d6d5d0')
fig.show()
pd.DataFrame({(lab, n): [gv.pct(a.at[y, v], b.at[y, v]) for y in (2021, 2025, 2030, 2050)]
              for v, lab in vars_.items() for n, (a, b) in parts.items()}, index=[2021, 2025, 2030, 2050]).T.round(3)

2021    2025    2030    2050
Real GDP                        S2 total     -1.311  -1.458  -1.831  -2.119
                                Switch only  -1.333  -1.446  -1.559  -1.741
                                Tax only      0.022  -0.012  -0.277  -0.385
Energy use, PJ                  S2 total      5.370   4.530   2.971   2.682
                                Switch only   5.377   4.891   4.819   4.633
                                Tax only     -0.007  -0.345  -1.763  -1.865
Real energy inputs (CGE volume) S2 total    -12.588 -13.571 -16.249 -16.173
                                Switch only -12.590 -12.932 -13.003 -12.819
                                Tax only      0.003  -0.734  -3.732  -3.848
CO2e emissions                  S2 total      5.188   3.381  -1.514  -1.907
                                Switch only   5.175   4.498   4.395   4.205
                                Tax only      0.012  -1.069  -5.660  -5.866

## Where the switch hits: energy use by purpose in 2021

Energy use `qEpj` (PJ) by purpose in 2021 (no tax yet): the zero-tax run against the baseline. The table below
lists the good-purpose pairs that rise most.

In [4]:
fb, fz = gv.flows(base, 2021, variant='shock_carbon_tax'), gv.flows(zero, 2021, variant='shock_carbon_tax')
t = gv.change_table(fb, fz, ('purpose',)).sort_values('change')
fig = go.Figure(go.Bar(x=t['change'], y=t.index, orientation='h',
                       marker=dict(color=['#1c5cab' if v < 0 else '#e34948' for v in t['change']]),
                       text=[f'{v:+.1f} PJ ({p:+.0f} %)' for v, p in zip(t['change'], t['% change'])],
                       textposition='outside', textfont=dict(color='#52514e'),
                       hovertemplate='%{y}: %{x:.2f} PJ<extra></extra>'))
fig.update_layout(template=gv.TEMPLATE, height=360, title='Change in energy use 2021 from the switch alone (PJ)',
                  xaxis_title='PJ')
fig.show()
gv.change_table(fb, fz, ('good', 'purpose')).sort_values('change', ascending=False).round(2).head(12)

baseline   shock  change  % change
good                purpose                                                 
Oil products        Transport               654.81  711.16   56.35      8.61
Bioenergy and waste Processes in the ETS     74.46   88.68   14.22     19.10
Coal                Processes, special       32.16   43.39   11.22     34.90
Bioenergy and waste Processes, special       70.32   76.70    6.38      9.08
Natural gas         Processes in the ETS     43.59   49.79    6.20     14.23
Electricity         Processes, special       47.02   52.20    5.17     11.00
Oil products        Processes, special      310.70  313.79    3.09      0.99
                    Processes in the ETS     17.07   19.65    2.58     15.11
Electricity         Heating                  50.14   52.60    2.46      4.90
Bioenergy and waste Heating                  42.55   44.43    1.88      4.41
Electricity         Processes, normal        34.15   35.84    1.70      4.97
District heat       Processes, normal        32.28   33.89    1.61      4.98

## Question for DREAM

Is the switch to the price-elastic branch in the CO2 tax script intended? If the tax should be measured from the
baseline as calibrated, either the shock keeps `d1switch_integrate_energy_technology = 1`, or the baseline is
re-solved with it at 0 first. The S3 runs (technology block on) are consistent.